# Linguistic-Marker Analysis of True Model Errors
### Arabic Sentiment Analysis (HARD) — Ensemble Learning Paper

This notebook reproduces the "Linguistic-Marker Analysis of True Model Errors" subsection
of the manuscript. It takes the annotated False Positive / False Negative spreadsheets
(reconciled noise annotation already applied), isolates the **true model errors**
(cases where annotators agreed the ground-truth label is correct, i.e. `AND == False`),
and tags each review with automated, transparent regex/keyword markers:

- `positive_title_negative_detail_structure` — short positive/neutral title + itemized complaint body (or mirror)
- `terse_short_review` — ≤8 Arabic content words
- `negation_heavy` — ≥4 negation particles
- `code_switching` — Latin-script sequences embedded in the Arabic text
- `gulf_dialect_marker` / `levantine_dialect_marker` — common regional lexical items
- `possible_irony_star_rating_vs_text` — star emoji alongside complaint text
- `unclassified` — no pattern detected (reported honestly, not forced into a category)

**Methodology note:** this is rule-based automated tagging, not a second round of manual
native-speaker annotation. It is a transparent, reproducible complement to the manual
noise-annotation reported elsewhere in the paper — see the manuscript text for the
methodology caveat.

**To run on Colab:**
1. Run the setup cell to install/import dependencies.
2. Upload `camelbert_seed42_FP_Annotated.xlsx` and `camelbert_seed42_FN_Annotated.xlsx`
   when prompted (or mount Google Drive and adjust the paths in the "Load data" cell).
3. Run all remaining cells top to bottom.


## 1. Setup

In [6]:
# Install dependencies (safe to re-run; no-ops if already satisfied)
!pip install -q openpyxl pandas


In [7]:
import re
import pandas as pd
import openpyxl
from collections import Counter

pd.set_option('display.max_colwidth', 120)


## 2. Locate input files (Google Drive)

**⚠️ Known data issue (found during testing):** as of this writing, the `.csv`
copy of `camelbert_seed42_FP_Annotated` in `EL4ASA/data/` on Drive is an
**incomplete/outdated export** -- it's missing "Noise Narrator 2" annotations
for ~36 of 260 reviews. The `.xlsx` version has the complete annotation and
matches the manuscript exactly. **Default below points at the `.xlsx` files
for this reason.** If you re-export a complete, up-to-date CSV to Drive,
switch `FP_PATH`/`FN_PATH` to the `.csv` paths (commented out below) --
the notebook's diagnostic check (Section 3) will tell you immediately if a
file you point it at is incomplete.

```
My Drive/EL4ASA/data/camelbert_seed42_FP_Annotated.xlsx
My Drive/EL4ASA/data/camelbert_seed42_FN_Annotated.xlsx
```

If your Drive layout or filenames differ, edit `EL4ASA_ROOT` / the paths below.

In [8]:
# --- Option A: Google Drive (default) ---
from google.colab import drive
drive.mount('/content/drive')

EL4ASA_ROOT = "/content/drive/MyDrive/EL4ASA"

# Using .xlsx by default -- see the data-completeness warning above.
FP_PATH = f"{EL4ASA_ROOT}/data/camelbert_seed42_FP_Annotated.xlsx"
FN_PATH = f"{EL4ASA_ROOT}/data/camelbert_seed42_FN_Annotated.xlsx"

# If you've re-exported complete, up-to-date CSVs to Drive, switch to these instead:
# FP_PATH = f"{EL4ASA_ROOT}/data/camelbert_seed42_FP_Annotated.csv"
# FN_PATH = f"{EL4ASA_ROOT}/data/camelbert_seed42_FN_Annotated.csv"

print(f"FP_PATH = {FP_PATH}")
print(f"FN_PATH = {FN_PATH}")

# --- Option B: direct upload (fallback -- uncomment if not using Drive,
#     or if running outside Colab / the files aren't in Drive) ---
# try:
#     from google.colab import files
#     print("Please upload camelbert_seed42_FP_Annotated.xlsx/csv and "
#           "camelbert_seed42_FN_Annotated.xlsx/csv")
#     uploaded = files.upload()
#     FP_PATH = list(uploaded.keys())[0]
#     FN_PATH = list(uploaded.keys())[1]
# except ImportError:
#     FP_PATH = "camelbert_seed42_FP_Annotated.xlsx"
#     FN_PATH = "camelbert_seed42_FN_Annotated.xlsx"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FP_PATH = /content/drive/MyDrive/EL4ASA/data/camelbert_seed42_FP_Annotated.xlsx
FN_PATH = /content/drive/MyDrive/EL4ASA/data/camelbert_seed42_FN_Annotated.xlsx


## 3. Load and clean the annotated data

**Important methodology note (corrected in this version):** the annotation protocol
uses two independent narrators (`Noise Narrator 1`, `Noise Narrator 2`) whose
agreement determines a three-way reconciled category:

- **`noise`** — both narrators agree the label is noise (ground truth is wrong)
- **`model_error`** — both narrators agree the label is correct (the model's
  prediction is the genuine error)
- **`ambiguous`** — narrators disagree; reconciled as "Not Sure" per the
  annotation protocol, i.e. genuinely unresolved, not a confirmed model error

We derive this directly from the two narrator columns rather than trusting any
single "reconciliation" column, because different exports of this data use
different column names for it (`AND`/`Disagreement` in one `.xlsx` export,
`XOR`/`Final Decision` in one `.csv` export, and no reconciliation column at
all in another `.csv` export) -- deriving it from the two raw narrator votes
is robust to all of these and was verified to reproduce the manuscript's
exact published noise/ambiguous percentages (49.2%/27.7% for FP, 32.6%/29.1%
for FN).

In [9]:
def load(path):
    """Load an annotated FP/FN file (.csv or .xlsx) and drop footer/summary rows."""
    if str(path).lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        wb = openpyxl.load_workbook(path, data_only=True)
        ws = wb[wb.sheetnames[0]]
        headers = [c.value for c in ws[1]]
        rows = list(ws.iter_rows(min_row=2, values_only=True))
        df = pd.DataFrame(rows, columns=headers)

    # Coerce 'index' to numeric regardless of how it was read; drop rows that
    # aren't a valid review index (this removes footer/summary rows).
    df['index'] = pd.to_numeric(df['index'], errors='coerce')
    df = df[df['index'].notna()]

    # Normalize the two narrator columns to real booleans regardless of source
    # representation (native bool, or "True"/"TRUE"/"False" strings from a
    # CSV round-trip).
    def to_bool(v):
        if isinstance(v, bool):
            return v
        if isinstance(v, str):
            s = v.strip().lower()
            if s == 'true':
                return True
            if s == 'false':
                return False
        return None

    df['Noise Narrator 1'] = df['Noise Narrator 1'].apply(to_bool)
    df['Noise Narrator 2'] = df['Noise Narrator 2'].apply(to_bool)
    n_missing_n1 = df['Noise Narrator 1'].isna().sum()
    n_missing_n2 = df['Noise Narrator 2'].isna().sum()
    df = df[df['Noise Narrator 1'].isin([True, False]) & df['Noise Narrator 2'].isin([True, False])]

    # Derive the reconciled 3-way category directly from the two narrator
    # votes -- robust to whichever (if any) auxiliary reconciliation column
    # a given export happens to include.
    def categorize(row):
        n1, n2 = row['Noise Narrator 1'], row['Noise Narrator 2']
        if n1 and n2:
            return 'noise'
        if not n1 and not n2:
            return 'model_error'
        return 'ambiguous'

    df['category'] = df.apply(categorize, axis=1)
    df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')
    df.attrs['n_missing_narrator1'] = n_missing_n1
    df.attrs['n_missing_narrator2'] = n_missing_n2
    return df.reset_index(drop=True)

fp = load(FP_PATH)
fn = load(FN_PATH)

print(f"FP clean rows: {len(fp)}")
print(fp['category'].value_counts())
print(f"\nFN clean rows: {len(fn)}")
print(fn['category'].value_counts())

# Diagnostic check against the manuscript's reported reconciled figures.
# FP: 260 total -- 128 noise (49.2%), 60 model_error (23.1%), 72 ambiguous (27.7%)
# FN: 141 total -- 46 noise (32.6%), 54 model_error (38.3%), 41 ambiguous (29.1%)
def diagnose(name, df_clean, expected_total, expected_noise, expected_model_error):
    counts = df_clean['category'].value_counts()
    ok = (len(df_clean) == expected_total
          and counts.get('noise', 0) == expected_noise
          and counts.get('model_error', 0) == expected_model_error)
    if ok:
        return True
    print(f"\n⚠️  {name} MISMATCH: got n={len(df_clean)} (expected {expected_total}), "
          f"noise={counts.get('noise',0)} (expected {expected_noise}), "
          f"model_error={counts.get('model_error',0)} (expected {expected_model_error})")
    print(f"   Rows dropped for missing narrator votes in the raw file: "
          f"Narrator1={df_clean.attrs.get('n_missing_narrator1','?')}, "
          f"Narrator2={df_clean.attrs.get('n_missing_narrator2','?')}")
    print(f"   This usually means the source file is an incomplete/outdated export. "
          f"Check for a more complete/recent version of this file before trusting "
          f"the results below (a .xlsx export was found complete when a .csv export "
          f"of the same data was not -- see Section 2 notes).")
    return False

fp_ok = diagnose("FP", fp, 260, 128, 60)
fn_ok = diagnose("FN", fn, 141, 46, 54)

if fp_ok and fn_ok:
    print("\n✅ Counts match the manuscript's reported reconciled noise/model-error/ambiguous annotation.")
else:
    print("\n❌ Proceeding with whatever data loaded, but results below will NOT match the "
          "manuscript until the mismatch above is resolved.")


FP clean rows: 260
category
noise          128
ambiguous       72
model_error     60
Name: count, dtype: int64

FN clean rows: 141
category
model_error    54
noise          46
ambiguous      41
Name: count, dtype: int64

✅ Counts match the manuscript's reported reconciled noise/model-error/ambiguous annotation.


## 4. Isolate true model errors

**Corrected definition:** true model errors are strictly rows where `category == 'model_error'`
(both narrators agree the label is correct, i.e. the model's prediction is the
genuine error) -- this **excludes** the `ambiguous` ("Not Sure" / disagreement)
cases, which are not confirmed model errors and should not be counted as such.

An earlier version of this notebook used `AND == False`, which incorrectly
included the ambiguous cases alongside genuine confirmed model errors,
inflating the true-model-error count (132 vs. the correct 60 for FP; 95 vs.
the correct 54 for FN).

In [10]:
fp_err = fp[fp['category'] == 'model_error'].copy()
fn_err = fn[fn['category'] == 'model_error'].copy()

print(f"FP true model errors (corrected): {len(fp_err)} of {len(fp)}  (expected: 60)")
print(f"FN true model errors (corrected): {len(fn_err)} of {len(fn)}  (expected: 54)")


FP true model errors (corrected): 60 of 260  (expected: 60)
FN true model errors (corrected): 54 of 141  (expected: 54)


## 5. Define the linguistic-marker classifier

In [11]:
QUOTE_TITLE = re.compile(r'^[“"]([^”"]+)[”"]\.?\s*')
LATIN = re.compile(r'[A-Za-z]{2,}')
GULF_DIALECT = re.compile(r'\b(وايد|مب|شلون|شنو|زين|هني|ليش|ابغى|ابى|احين|الحين)\b')
LEVANTINE_DIALECT = re.compile(r'\b(كتير|هيك|شو|منيح|هلق|بدي)\b')
STAR_EMOJI = re.compile(r'⭐|★')

COMPLAINT_MARKERS = re.compile(
    r'(عدم وجود|لا يوجد|لايوجد|غير مرضي|غير نظيف|غير جيد|غير واضح|لم يكن|لم يعمل|لايعمل|رفض|تاخير|تاخر|'
    r'ازعاج|مشكله|مشكلة|سيئ|سيئه|ضيق|ضيقه|صغير|صغيره|متسخ|معطل|بطيء|ضعيف|ضعيفه|غلاء|غالي|زياده|زيادة|قديم|قديمه|بايخ|'
    r'متهالك|باليه|قذر|رائحه كريهه|رائحة كريهة|شكوي|شكوى|رديء|اشكال|بارد|بعيد|بعيده|صعوبه|صعوبة|'
    r'لم يتم|لا توجد|لاتوجد|صدا)'
)
NEGATIVE_TITLE_MARKERS = re.compile(r'(سيئ|رديء|لا انصح|غالي|مخيب|سيء|فشل|ندم)')
NEGATION = re.compile(r'\b(لا|لم|لن|ليس|غير|عدم|ما|مافي|مافيه|لايوجد|بدون)\b')


def word_count(text):
    return len(re.findall(r'[\u0600-\u06FF]+', text))


def classify(text):
    """Tag a single review with linguistic-marker categories (not mutually exclusive)."""
    tags = []
    m = QUOTE_TITLE.match(text.strip())
    title = m.group(1) if m else ''
    body = text[m.end():] if m else text

    title_has_complaint = bool(NEGATIVE_TITLE_MARKERS.search(title))
    body_complaint_count = len(COMPLAINT_MARKERS.findall(body))

    if title and not title_has_complaint and body_complaint_count >= 1:
        tags.append('positive_title_negative_detail_structure')
    elif body_complaint_count >= 2:
        tags.append('itemized_complaint_list')

    if len(NEGATION.findall(text)) >= 4:
        tags.append('negation_heavy')
    if LATIN.search(text):
        tags.append('code_switching')
    if GULF_DIALECT.search(text):
        tags.append('gulf_dialect_marker')
    if LEVANTINE_DIALECT.search(text):
        tags.append('levantine_dialect_marker')
    if word_count(text) <= 8:
        tags.append('terse_short_review')
    if STAR_EMOJI.search(text) and body_complaint_count >= 1:
        tags.append('possible_irony_star_rating_vs_text')

    if not tags:
        tags.append('unclassified')
    return tags


## 6. Apply the classifier and tally results

In [12]:
fp_err['tags'] = fp_err['review'].apply(classify)
fn_err['tags'] = fn_err['review'].apply(classify)

def tag_table(df, label):
    counts = Counter(t for tags in df['tags'] for t in tags)
    n = len(df)
    rows = [{'pattern': k, 'count': v, 'pct': round(v / n * 100, 1)} for k, v in counts.most_common()]
    out = pd.DataFrame(rows)
    print(f"\n{label} true-model-errors (n={n}):")
    display(out)
    return out

fp_table = tag_table(fp_err, "FP")
fn_table = tag_table(fn_err, "FN")



FP true-model-errors (n=60):


,pattern,count,pct
0,positive_title_negative_detail_structure,34,56.7
1,unclassified,16,26.7
2,terse_short_review,9,15.0
3,gulf_dialect_marker,2,3.3
4,code_switching,2,3.3
5,levantine_dialect_marker,1,1.7
6,possible_irony_star_rating_vs_text,1,1.7
7,negation_heavy,1,1.7



FN true-model-errors (n=54):


,pattern,count,pct
0,positive_title_negative_detail_structure,24,44.4
1,unclassified,21,38.9
2,terse_short_review,12,22.2
3,negation_heavy,1,1.9


## 7. Reproduce the manuscript's headline finding

Expected (corrected methodology, strict `model_error` subset only):
**56.7%** of FP true model errors (n=60) and **44.4%** of FN true model errors
(n=54) show the `positive_title_negative_detail_structure` pattern.

In [13]:
fp_structural = fp_err['tags'].apply(lambda t: 'positive_title_negative_detail_structure' in t).sum()
fn_structural = fn_err['tags'].apply(lambda t: 'positive_title_negative_detail_structure' in t).sum()

print(f"FP: {fp_structural}/{len(fp_err)} = {fp_structural/len(fp_err)*100:.1f}% (manuscript: 56.7%)")
print(f"FN: {fn_structural}/{len(fn_err)} = {fn_structural/len(fn_err)*100:.1f}% (manuscript: 44.4%)")


FP: 34/60 = 56.7% (manuscript: 56.7%)
FN: 24/54 = 44.4% (manuscript: 44.4%)


## 8. Inspect example reviews per category (optional, for qualitative spot-checks)

In [14]:
category = 'positive_title_negative_detail_structure'  # change to inspect other categories
sample = fp_err[fp_err['tags'].apply(lambda t: category in t)].sample(
    min(5, len(fp_err)), random_state=1
)
for _, row in sample.iterrows():
    print('---')
    print(f"confidence={row['confidence']:.3f}")
    print(row['review'][:300])


---
confidence=0.963
“موقع علي الجبل”. هدوء المكان. عدم ذكر ان مكان الفندق علي تله عاليه وصعوبه الصعود للمكان مع عدم تواقر وسيله مواصلات.
---
confidence=0.950
“تجربه مقبوله”. نظافه الاثاثهدوء تام. عدم وجود عماله للتنظيف خاصه ان زمن وصولنا كان بالعيد وكانت العماله في اجازه مما تسبب في ركن الاوساخ بين ممرات الغرف واضافه الي ذلك اضطريت لحمل الحقائب بنفسي لنفس السبب وهو اجازه العماله يوم العيدعدم وجود مياه للشرب في المني بار في الغرفهعدم وجود مناديل في بعض ال
---
confidence=0.980
“اعطني المال ثم اخدمك”. الموقع و الاثاث والنظافه. اولا اخذ عربون.ثانيا لم يكن نفس المبلغ الذي تم عرضه علي الموقع تفاجأت باضافه رسوم اضافيه.ثالثا وصلنا متعبين من السفر قبل موعد الدخول ولم يسمح لنا بدخول الشقه الا بعد اخذ مبلغ نصف يوم مقابل راحتنا انا واطفالي لعدد الساعات المتبقيه.
---
confidence=0.905
“الاقامه في ايبيس”. النظافه. موقعه + ضيق الغرف + جاي تعبان يقولي مافيه احد يخدمك اخدم نفسك وطلع او نزل الشنط هذا فندق رجال اعمال + السرير ضيق
---
confidence=0.630
“Cap ten ali. ✈️”. الشكر للاخ احمد الهنيدي والسيد. اك

## 9. Export tagged data

In [15]:
# Save locally (always)
fp_err[['index', 'review', 'confidence', 'tags']].to_csv(
    'fp_linguistic_tags.csv', index=False, encoding='utf-8-sig'
)
fn_err[['index', 'review', 'confidence', 'tags']].to_csv(
    'fn_linguistic_tags.csv', index=False, encoding='utf-8-sig'
)
print("Saved fp_linguistic_tags.csv and fn_linguistic_tags.csv (local runtime)")

# Also save back to the EL4ASA Drive structure, alongside the other error-analysis
# outputs (mirrors the existing results/error_analysis/ convention in the project)
import os
DRIVE_OUT_DIR = f"{EL4ASA_ROOT}/results/error_analysis"
try:
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    fp_err[['index', 'review', 'confidence', 'tags']].to_csv(
        f'{DRIVE_OUT_DIR}/fp_linguistic_tags.csv', index=False, encoding='utf-8-sig'
    )
    fn_err[['index', 'review', 'confidence', 'tags']].to_csv(
        f'{DRIVE_OUT_DIR}/fn_linguistic_tags.csv', index=False, encoding='utf-8-sig'
    )
    print(f"Also saved to Drive: {DRIVE_OUT_DIR}/")
except NameError:
    print("EL4ASA_ROOT not defined (not using Drive) -- skipped Drive save.")

# On Colab, also offer a direct download of the local copies:
try:
    from google.colab import files
    files.download('fp_linguistic_tags.csv')
    files.download('fn_linguistic_tags.csv')
except ImportError:
    pass


Saved fp_linguistic_tags.csv and fn_linguistic_tags.csv (local runtime)
Also saved to Drive: /content/drive/MyDrive/EL4ASA/results/error_analysis/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>